# EVE 310 - Lab 08: Classification thresholds — activity

**Module 3 | 10/15/2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyanRevolter/eve310-fall-2026/blob/main/labs/lab08-classification-thresholds/notebooks/lab08-activity.ipynb)

## Learning objectives

By the end of this lab you will be able to:

1. Count the four classification outcomes at a new threshold with a loop and an if, and compute precision and recall from them
2. Sweep a grid of thresholds with a for loop and plot precision and recall for 643 wells
3. Choose a threshold from a stated requirement

## Before you start

1. Click **Copy to Drive** at the top of this window and work in the copy that opens. Colab throws away anything you did not copy when the runtime ends.
2. Run the setup cell below before anything else. It downloads `TCE_LongIsland.csv` into `DATA_DIR` and creates `FIGURES_DIR`.
3. Work down the notebook in order. Replace every `--` with your code and keep the variable names exactly as given. Three short questions: one counting loop, one sweep, one choice.
4. When you are done: **Runtime > Restart session and run all**, then **File > Download > Download .ipynb**, and upload that file to Gradescope. The autograder gives you feedback on each question, but **this submission is not graded** and does not count toward your final grade. The Lab 8 quiz on Canvas is what counts, and it asks about what you compute here, so finish the activity before you take it.

## 0. Setup

In [ ]:
# EVE 310 setup - run this cell first, every time you open this notebook.
# It downloads this lab's data file into a "data" folder in your Colab session.
import pathlib
import urllib.request

LAB = "lab08-classification-thresholds"
DATA_FILES = ["TCE_LongIsland.csv"]

DATA_DIR = pathlib.Path("data")
FIGURES_DIR = pathlib.Path("figures")
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

BASE_URL = f"https://raw.githubusercontent.com/ThyanRevolter/eve310-fall-2026/main/labs/{LAB}/data"
for name in DATA_FILES:
    if not (DATA_DIR / name).exists():
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", DATA_DIR / name)

print(f"Ready. {len(DATA_FILES)} data file(s) in {DATA_DIR.resolve()}")

**The data (run as is)**

All 643 wells of the Long Island TCE study from lecture, one row per well: `density`, the population density near the well in persons per acre; `detect`, 1 if TCE above 3 ppb was measured in the well and 0 if not; and `prob`, the probability of a detection from the logistic regression of Lecture 16, which was fit on these same wells. `y_true` and `y_prob` are the same two arrays as in the tutorial, 643 values long instead of 10.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score

tce = pd.read_csv(DATA_DIR / 'TCE_LongIsland.csv')
y_true = tce['detect'].to_numpy()   # 1 = TCE above 3 ppb was measured in the well
y_prob = tce['prob'].to_numpy()     # the model's probability of a detection

print(len(y_true), 'wells,', y_true.sum(), 'true detections')
tce.head()

## Your turn

**Question 1 — The four outcomes at a threshold of 0.3**

The default threshold is 0.5. Use **0.3** instead: predict the labels, then count the four outcomes with one loop and an `if` / `elif` chain, as in the tutorial.

- `y_pred`: the predicted labels at threshold 0.3 (`True` / `False`, one per well)
- `tp`, `fp`, `fn`, `tn`: the four counts
- `precision_hand`, `recall_hand`: precision and recall from the four counts, by their definitions (no sklearn here)

The check lines make sure the four counts cover every well and compare two of them with the array shortcut.

In [ ]:
threshold_1 = 0.3

y_pred = --

tp = 0
fp = 0
fn = 0
tn = 0

for i in range(len(y_true)):
    if --:
        tp = tp + 1
    elif --:
        fp = fp + 1
    elif --:
        fn = fn + 1
    else:
        tn = tn + 1

precision_hand = --
recall_hand = --

print('TP:', tp, ' FP:', fp, ' FN:', fn, ' TN:', tn)
print('Precision:', round(precision_hand, 3), ' Recall:', round(recall_hand, 3))
print('Covers every well?', tp + fp + fn + tn == len(y_true))
print('Matches the array shortcut?', tp == ((y_true == 1) & (y_pred == 1)).sum(), fn == ((y_true == 1) & (y_pred == 0)).sum())

At this threshold, which is larger: the number of false alarms or the number of missed detections? For a drinking-water screening, would you rather raise or lower the threshold, and why?

*Your answer:*

**Question 2 — Sweep the threshold**

Try 21 thresholds, 0 to 1 in steps of 0.05 (the grid is given). Fill `precision` and `recall` with one loop, using `precision_score` with `zero_division=1` and `recall_score`, then plot both against the threshold.

- `precision`, `recall`: arrays of 21 values, one per threshold
- `fig`: the figure, precision in blue and recall in red, with the labels, title and legend given

The check line confirms that slot 6 of your arrays (threshold 0.3) agrees with Question 1.

In [ ]:
threshold = np.arange(0, 1.01, 0.05)
precision = np.zeros(len(threshold))
recall = np.zeros(len(threshold))

for i in range(len(threshold)):
    y_pred = --
    precision[i] = --
    recall[i] = --

fig, ax = plt.subplots()
--
--
ax.set_xlabel('Threshold')
ax.set_ylabel('Precision, recall')
ax.set_title('TCE detection: selecting the threshold')
ax.legend(['Precision', 'Recall'])
fig.savefig(FIGURES_DIR / 'lab08_threshold.png', dpi=200)

print('threshold  precision  recall')
for i in range(len(threshold)):
    print(f'{threshold[i]:9.2f}  {precision[i]:9.3f}  {recall[i]:6.3f}')
print('Slot 6 is threshold', round(threshold[6], 2), '- matches Question 1?', np.isclose(precision[6], precision_hand), np.isclose(recall[6], recall_hand))

As the threshold rises, one of the two curves can never go up. Which one, and why? What happens to the other one between 0.70 and 0.85, and what does the `zero_division=1` convention have to do with it?

*Your answer:*

**Question 3 — Choose a threshold from a requirement**

The county will re-test every well the model flags. It wants to catch **at least 80%** of the contaminated wells (a recall of 0.8 or more) and, among the thresholds that manage that, to send as few false alarms as possible, that is, the one with the **highest precision**. Find that threshold in your grid with the "largest so far" pattern from Lab 7: loop over the slots, and whenever a slot meets the recall requirement **and** beats the best precision seen so far, remember its precision and its threshold.

- `chosen_threshold`: the threshold from your grid that meets the requirement
- `n_flagged`: how many of the 643 wells the model flags at that threshold

The check line confirms that the recall at `chosen_threshold` is at least 0.8.

In [ ]:
best_precision = 0
chosen_threshold = threshold[0]

for i in range(len(threshold)):
    if --:
        best_precision = precision[i]
        chosen_threshold = threshold[i]

n_flagged = --

print('Chosen threshold:', round(chosen_threshold, 2), ' precision there:', round(best_precision, 3))
print(n_flagged, 'wells flagged out of', len(y_true))
print('Recall there at least 0.8?', recall_score(y_true, y_prob > chosen_threshold) >= 0.8)

At your chosen threshold, roughly what fraction of the 643 wells gets re-tested, and what fraction of the flagged wells are false alarms? Is a model that uses population density alone good enough for this job?

*Your answer:*

**Quiz scratch space**

The Canvas quiz asks about the values above. Use the cell below during the quiz.

**DO NOT OVERWRITE ANY VARIABLES ABOVE THIS CELL. CREATE NEW VARIABLES AS NEEDED.**

In [ ]:
# Use this cell for quiz calculations

## Wrap-up

**Runtime > Restart session and run all**, confirm every cell runs top to bottom, then **File > Download > Download .ipynb** and upload it to Gradescope for feedback. The submission is not graded. Keep this notebook open for the Canvas quiz.